# UniChem Adapter Example
Demonstrates searching and retrieving chemical compound cross-references using the UniChemAdapter.

## Overview
The UniChemAdapter integrates with EMBL-EBI's UniChem service to provide cross-referencing between chemical compound identifiers across different databases. This adapter uses the `bioservices.UniChem` library for reliable API access.

### Features Demonstrated:
- Compound search by various identifiers (InChI, InChIKey, UCI, source IDs)
- Cross-reference retrieval across chemical databases
- Source information and connectivity searches
- Integration with the knowledge graph system

In [17]:
# Import required libraries
from aid_pais_knowledgegraph.knowledge_lookup.adapters.unichem_adapter import UniChemAdapter
from aid_pais_knowledgegraph.knowledge_lookup.models import LookupConfig
from aid_pais_knowledgegraph.knowledge_lookup import init_cache
from aid_pais_knowledgegraph.knowledge_lookup.utils.timing_utils import (
    time_async_operation,
    format_timing_comparison,
    timing_decorator,
    benchmark_decorator
)
import asyncio
import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Initialize caching system (optional - enables disk persistence)
cache = init_cache(
    memory_max_size=1000,  # Max 1000 entries in memory
    disk_cache_dir="./unichem_cache",  # Enable disk caching
    disk_max_size=10000,  # Max 10000 entries on disk
    default_ttl=3600,  # 1 hour default TTL
)

# Initialize adapter
config = LookupConfig()
adapter = UniChemAdapter(config)

print(f"UniChemAdapter initialized. Available: {adapter.is_available()}")
print(f"Source: {adapter.get_source()}")
print(f"Cache initialized with disk persistence")

UniChemAdapter initialized. Available: True
Source: KnowledgeSource.UNICHEM
Cache initialized with disk persistence


In [18]:
# Test caching with source information (cached for 24 hours)
print("\n=== Testing Source Information Caching ===")

# First call - should be a cache miss
time1, sources1 = await time_async_operation(
    lambda: adapter.search_concepts("CHEMBL25", limit=5),
    "First call (cache miss - API fetch)"
)

# Second call - should be a cache hit
time2, sources2 = await time_async_operation(
    lambda: adapter.search_concepts("CHEMBL25", limit=5),
    "Second call (cache hit - fast retrieval)"
)

# Display timing comparison
print(format_timing_comparison(time1, time2, "API call", "Cache hit"))
print(f"Sources found: {len(sources1) if sources1 else 0}")


=== Testing Source Information Caching ===


INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.unichem_adapter:UniChem search for 'CHEMBL25' returned 1 concepts


First call (cache miss - API fetch):
  Duration: 0.734554 seconds
  Performance: 1.4 ops/sec


INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.unichem_adapter:UniChem search for 'CHEMBL25' returned 1 concepts


Second call (cache hit - fast retrieval):
  Duration: 0.730198 seconds
  Performance: 1.4 ops/sec
⏱️  Timing Comparison:
  API call: 0.734554s
  Cache hit: 0.730198s
  Speedup: 1.01x
   📊 Minor performance improvement.
Sources found: 1


## Basic Compound Search
Let's start by searching for compounds using different types of identifiers.

In [19]:
# Helper function to run async operations
async def run_async(func):
    return await func

# Search for aspirin by ChEMBL ID
print("=== Searching for Aspirin (CHEMBL25) ===")
results = await adapter.search_concepts("CHEMBL25", limit=5)
for concept in results:
    print(f"ID: {concept.primary_id}")
    print(f"Label: {concept.primary_label}")
    print(f"Type: {concept.concept_type}")
    print(f"Identifiers: {len(concept.identifiers)}")
    print(concept)

=== Searching for Aspirin (CHEMBL25) ===


INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.unichem_adapter:UniChem search for 'CHEMBL25' returned 1 concepts


ID: 161671
Label: UCI_161671
Type: ConceptType.CHEMICAL
Identifiers: 52
UnifiedConcept(primary_id='161671', primary_label='UCI_161671', concept_type=<ConceptType.CHEMICAL: 'chemical'>, identifiers=[ConceptIdentifier(source=<KnowledgeSource.UNICHEM: 'unichem'>, identifier='161671', label='UniChem Compound Identifier', url='https://www.ebi.ac.uk/unichem/compounds/161671'), ConceptIdentifier(source=<KnowledgeSource.UNICHEM: 'unichem'>, identifier='CHEMBL25', label='chembl ID', url='https://www.ebi.ac.uk/chembldb/compound/inspect/CHEMBL25'), ConceptIdentifier(source=<KnowledgeSource.UNICHEM: 'unichem'>, identifier='DB00945', label='drugbank ID', url='http://www.drugbank.ca/drugs/DB00945'), ConceptIdentifier(source=<KnowledgeSource.UNICHEM: 'unichem'>, identifier='AIN', label='pdb ID', url='http://www.ebi.ac.uk/pdbe-srv/pdbechem/chemicalCompound/show/AIN'), ConceptIdentifier(source=<KnowledgeSource.UNICHEM: 'unichem'>, identifier='4139', label='gtopdb ID', url='http://www.guidetopharmacolog

In [20]:
# Search by InChIKey
from nltk import pr


print("=== Searching by InChIKey (Aspirin) ===")
inchikey = "BSYNRYMUTXBXSQ-UHFFFAOYSA-N"  # Aspirin InChIKey
results = await adapter.search_concepts(inchikey, limit=3)
for concept in results:
    print(f"ID: {concept.primary_id}")
    print(f"Label: {concept.primary_label}")
    print(f"Categories: {concept.categories}")
    print(concept)
    print("---")

=== Searching by InChIKey (Aspirin) ===


INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.unichem_adapter:UniChem search for 'BSYNRYMUTXBXSQ-UHFFFAOYSA-N' returned 1 concepts


ID: 161671
Label: UCI_161671
Categories: ['chembl', 'drugbank', 'pdb', 'gtopdb', 'pubchem_dotf', 'kegg_ligand', 'chebi', 'zinc', 'emolecules', 'atlas', 'atlas', 'fdasrs', 'surechembl', 'pharmgkb', 'hmdb', 'selleck', 'pubchem_tpharma', 'pubchem', 'mcule', 'nmrshiftdb2', 'lincs', 'actor', 'actor', 'nikkaji', 'bindingdb', 'comptox', 'drugcentral', 'brenda', 'brenda', 'brenda', 'brenda', 'brenda', 'brenda', 'chemicalbook', 'chemicalbook', 'dailymed', 'clinicaltrials', 'clinicaltrials', 'clinicaltrials', 'clinicaltrials', 'clinicaltrials', 'clinicaltrials', 'clinicaltrials', 'clinicaltrials', 'rxnorm', 'rxnorm', 'rxnorm', 'rxnorm', 'MedChemExpress', 'probes_and_drugs', 'CCDC']
UnifiedConcept(primary_id='161671', primary_label='UCI_161671', concept_type=<ConceptType.CHEMICAL: 'chemical'>, identifiers=[ConceptIdentifier(source=<KnowledgeSource.UNICHEM: 'unichem'>, identifier='161671', label='UniChem Compound Identifier', url='https://www.ebi.ac.uk/unichem/compounds/161671'), ConceptIdentifier

In [21]:
# Search by UCI (UniChem Identifier)
print("=== Searching by UCI ===")
results = await adapter.search_concepts("1", limit=3)  # UCI 1
for concept in results:
    print(f"ID: {concept.primary_id}")
    print(f"Label: {concept.primary_label}")
    print("---")

=== Searching by UCI ===


INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.unichem_adapter:UniChem search for '1' returned 2 concepts


ID: 1
Label: UCI_1
---
ID: 32016566
Label: UCI_32016566
---


## Cross-Reference Retrieval
One of the most powerful features of UniChem is getting cross-references across different chemical databases.

In [22]:

print(f"UniChemAdapter re-initialized. Available: {adapter.is_available()}")

# Get cross-references for aspirin
print("=== Cross-references for Aspirin (CHEMBL25) ===")
xrefs = await adapter.get_cross_references("CHEMBL25")
print(f"Type of xrefs: {type(xrefs)}")
print(f"Sample xrefs content: {list(xrefs.items())[:2] if xrefs else 'Empty'}")
print(f"Found cross-references in {len(xrefs)} sources:")
for source, compounds in xrefs.items():
    print(f"\n{source}:")
    for compound in compounds[:2]:  # Show first 2 per source
        print(f"  ID: {compound['id']}")
        print(f"  URL: {compound['url']}")
    if len(compounds) > 2:
        print(f"  ... and {len(compounds) - 2} more")
    print()

UniChemAdapter re-initialized. Available: True
=== Cross-references for Aspirin (CHEMBL25) ===
Type of xrefs: <class 'dict'>
Sample xrefs content: [('chembl', [{'id': 'CHEMBL25', 'url': 'https://www.ebi.ac.uk/chembldb/compound/inspect/CHEMBL25'}]), ('drugbank', [{'id': 'DB00945', 'url': 'http://www.drugbank.ca/drugs/DB00945'}])]
Found cross-references in 33 sources:

chembl:
  ID: CHEMBL25
  URL: https://www.ebi.ac.uk/chembldb/compound/inspect/CHEMBL25


drugbank:
  ID: DB00945
  URL: http://www.drugbank.ca/drugs/DB00945


pdb:
  ID: AIN
  URL: http://www.ebi.ac.uk/pdbe-srv/pdbechem/chemicalCompound/show/AIN


gtopdb:
  ID: 4139
  URL: http://www.guidetopharmacology.org/GRAC/LigandDisplayForward?ligandId=4139


pubchem_dotf:
  ID: 24714725
  URL: http://pubchem.ncbi.nlm.nih.gov/substance/24714725


kegg_ligand:
  ID: C01405
  URL: http://www.genome.jp/dbget-bin/www_bget?C01405


chebi:
  ID: 15365
  URL: http://www.ebi.ac.uk/chebi/searchId.do?chebiId=CHEBI%3A15365


zinc:
  ID: ZINC000

## Source Information
UniChem integrates data from many different chemical databases. Let's explore the available sources.

In [23]:
# Get all available sources
print("=== Available UniChem Sources ===")
sources = await adapter.get_sources()
print(f"Total sources: {len(sources)}")

# Show first 10 sources
for source in sources[:10]:
    print(f"ID: {source.get('sourceID')} - {source.get('name')} ({source.get('nameLong')})")
    print(f"Compounds: {source.get('UCICount', 'N/A')}")
    print("---")

=== Available UniChem Sources ===
Total sources: 41
ID: 1 - chembl (ChEMBL)
Compounds: 2473434
---
ID: 2 - drugbank (DrugBank)
Compounds: 11919
---
ID: 3 - pdb (PDBe (Protein Data Bank Europe))
Compounds: 43246
---
ID: 4 - gtopdb (Guide to Pharmacology)
Compounds: 8411
---
ID: 5 - pubchem_dotf (PubChem ('Drugs of the Future' subset))
Compounds: 5646
---
ID: 6 - kegg_ligand (KEGG (Kyoto Encyclopedia of Genes and Genomes) Ligand)
Compounds: 14033
---
ID: 7 - chebi (ChEBI (Chemical Entities of Biological Interest).)
Compounds: 142602
---
ID: 8 - nih_ncc (NIH Clinical Collection)
Compounds: 719
---
ID: 9 - zinc (ZINC)
Compounds: 16886865
---
ID: 10 - emolecules (eMolecules)
Compounds: 5168336
---


In [24]:
# Get detailed information about a specific source
print("=== Detailed Source Information (ChEMBL) ===")
source_details = await adapter.get_source_info_by_id(1)  # ChEMBL is source ID 1
if source_details:
    print(f"Name: {source_details.get('name')}")
    print(f"Full Name: {source_details.get('nameLong')}")
    print(f"Description: {source_details.get('description')}")
    print(f"URL: {source_details.get('srcUrl')}")
    print(f"Last Updated: {source_details.get('lastUpdated')}")
    print(f"Total Compounds: {source_details.get('UCICount')}")
else:
    print("Could not retrieve source details")

=== Detailed Source Information (ChEMBL) ===
Name: chembl
Full Name: ChEMBL
Description: A database of bioactive drug-like small molecules and bioactivities abstracted from the scientific literature.
URL: https://www.ebi.ac.uk/chembl/
Last Updated: 2024-12-11
Total Compounds: 2473434


## Connectivity Search
Find compounds with similar chemical structure/connectivity.

In [25]:
# Find compounds with connectivity similar to aspirin
print("=== Compounds with Similar Connectivity to Aspirin ===")
# Note: UniChem connectivity search requires specific parameters
# For demonstration, we'll show connectivity info for aspirin instead
connectivity_info = await adapter.get_connectivity("CHEMBL25", "chembl")
if connectivity_info:
    print("Connectivity information retrieved for aspirin")
    print(f"Keys: {list(connectivity_info.keys()) if isinstance(connectivity_info, dict) else 'Not a dict'}")
else:
    print("Could not retrieve connectivity information")
    print("This feature requires specific UniChem API parameters")

=== Compounds with Similar Connectivity to Aspirin ===
Connectivity information retrieved for aspirin
Keys: ['response', 'searchedCompound', 'sources', 'totalCompounds', 'totalSources']
Connectivity information retrieved for aspirin
Keys: ['response', 'searchedCompound', 'sources', 'totalCompounds', 'totalSources']


In [26]:
connectivity_info

{'response': 'Success',
 'searchedCompound': {'inchi': 'InChI=1S/C9H8O4/c1-6(10)13-8-5-3-2-4-7(8)9(11)12/h2-5H,1H3,(H,11,12)',
  'standardInchiKey': 'BSYNRYMUTXBXSQ-UHFFFAOYSA-N',
  'uci': 161671},
 'sources': [{'baseIDURLAvailable': True,
   'comparison': {'HAtoms': True,
    'charge': True,
    'connectivity': True,
    'formula': True,
    'isotope': True,
    'isotopicExchangeableH': True,
    'protonation': True,
    'stereoDbond': True,
    'stereoSp3': True,
    'stereoSp3Inverted': True,
    'stereoType': True},
   'compoundId': 'CHEMBL25',
   'id': 1,
   'longName': 'ChEMBL',
   'shortName': 'chembl',
   'typeOfSearch': 'match',
   'url': 'https://www.ebi.ac.uk/chembldb/compound/inspect/CHEMBL25'},
  {'baseIDURLAvailable': True,
   'comparison': {'HAtoms': True,
    'charge': True,
    'connectivity': True,
    'formula': True,
    'isotope': True,
    'isotopicExchangeableH': True,
    'protonation': True,
    'stereoDbond': True,
    'stereoSp3': True,
    'stereoSp3Inverted

## Integration with Knowledge Graph
The UniChemAdapter integrates seamlessly with the broader knowledge graph system.

In [27]:
# Get detailed concept information
print("=== Detailed Concept Information ===")
concept = await adapter.get_concept_details("1")  # UCI 1
if concept:
    print(f"Primary ID: {concept.primary_id}")
    print(f"Primary Label: {concept.primary_label}")
    print(f"Concept Type: {concept.concept_type}")
    print(f"Confidence Score: {concept.confidence_score}")
    print(f"Categories: {concept.categories}")

    print(f"\nIdentifiers ({len(concept.identifiers)}):")
    for identifier in concept.identifiers[:10]:  # Show first 10
        print(f"  - {identifier.source}: {identifier.identifier} ({identifier.label})")

=== Detailed Concept Information ===
Primary ID: 1
Primary Label: UCI_1
Concept Type: ConceptType.CHEMICAL
Confidence Score: 0.9
Categories: ['chembl', 'zinc', 'surechembl', 'pubchem']

Identifiers (5):
  - KnowledgeSource.UNICHEM: 1 (UniChem Compound Identifier)
  - KnowledgeSource.UNICHEM: CHEMBL172985 (chembl ID)
  - KnowledgeSource.UNICHEM: ZINC000028009973 (zinc ID)
  - KnowledgeSource.UNICHEM: SCHEMBL6412790 (surechembl ID)
  - KnowledgeSource.UNICHEM: 18325210 (pubchem ID)


In [28]:
# Demonstrate RDF generation capability (if integrated with other adapters)
print("=== RDF/Turtle Generation Example ===")
if concept:
    # This would typically be done through the knowledge graph integration
    print("# Example RDF/Turtle output for UniChem compound:")
    print(f"<https://www.ebi.ac.uk/unichem/compoundsources/{concept.primary_id}>")
    print(f'    rdfs:label "{concept.primary_label}" ;')
    print(f'    aidpais:conceptType "{concept.concept_type.value}" ;')
    for identifier in concept.identifiers[:3]:
        if identifier.source.value == "unichem":
            continue  # Skip the primary UniChem identifier
        print(f'    aidpais:hasIdentifier "{identifier.identifier}" ;')
    print("    .")

=== RDF/Turtle Generation Example ===
# Example RDF/Turtle output for UniChem compound:
<https://www.ebi.ac.uk/unichem/compoundsources/1>
    rdfs:label "UCI_1" ;
    aidpais:conceptType "chemical" ;
    .


## Advanced Usage Examples

In [29]:
# Search for multiple compounds at once
print("=== Batch Search Example ===")
test_compounds = ["CHEMBL25", "CHEMBL3", "CHEMBL2"]  # Aspirin, Caffeine, Theophylline

for compound_id in test_compounds:
    try:
        xrefs = await adapter.get_cross_references(compound_id)
        chembl_count = len([c for c in xrefs.get('chembl', [])])
        pubchem_count = len([c for c in xrefs.get('pubchem', [])])
        drugbank_count = len([c for c in xrefs.get('drugbank', [])])

        print(f"{compound_id}: ChEMBL({chembl_count}) PubChem({pubchem_count}) DrugBank({drugbank_count})")
    except Exception as e:
        print(f"{compound_id}: Error - {e}")

=== Batch Search Example ===
CHEMBL25: ChEMBL(1) PubChem(1) DrugBank(1)
CHEMBL25: ChEMBL(1) PubChem(1) DrugBank(1)
CHEMBL3: ChEMBL(1) PubChem(1) DrugBank(1)
CHEMBL3: ChEMBL(1) PubChem(1) DrugBank(1)
CHEMBL2: ChEMBL(1) PubChem(1) DrugBank(1)
CHEMBL2: ChEMBL(1) PubChem(1) DrugBank(1)


In [30]:
# Demonstrate error handling
print("=== Error Handling Example ===")
try:
    # Try to search for a non-existent compound
    results = await adapter.search_concepts("NONEXISTENT_COMPOUND_12345")
    print(f"Results found: {len(results)}")
except Exception as e:
    print(f"Handled error gracefully: {e}")

=== Error Handling Example ===


INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.unichem_adapter:UniChem search for 'NONEXISTENT_COMPOUND_12345' returned 0 concepts


Results found: 0


# RDF Knowledge Graph Integration

Now that you have UniChem compound data, here's how to integrate it into your RDF knowledge graph. The AID-PAIS knowledge graph uses Turtle (.ttl) format with custom namespaces for symptoms and vocabularies.

In [31]:
# RDF Integration Setup
import rdflib
from rdflib import Graph, URIRef, Literal, Namespace, RDF, RDFS, OWL
from pathlib import Path
from typing import List
from aid_pais_knowledgegraph.knowledge_lookup.models import UnifiedConcept, KnowledgeSource

# Define namespaces (following your existing KG structure)
AIDPAIS = Namespace("http://www.aid-pais-kg.org/")
CHEMBL = Namespace("https://www.ebi.ac.uk/chembl/compound/")
PUBCHEM = Namespace("https://pubchem.ncbi.nlm.nih.gov/compound/")
DRUGBANK = Namespace("https://www.drugbank.ca/drugs/")
UNICHEM = Namespace("https://www.ebi.ac.uk/unichem/compoundsources/")

def create_rdf_graph_for_compounds(concepts: List[UnifiedConcept]) -> Graph:
    """
    Convert UniChem UnifiedConcept objects to RDF triples.

    Args:
        concepts: List of UnifiedConcept objects from UniChem searches

    Returns:
        RDFLib Graph containing the compound data
    """
    g = Graph()

    # Bind namespaces
    g.bind("aidpais", AIDPAIS)
    g.bind("chembl", CHEMBL)
    g.bind("pubchem", PUBCHEM)
    g.bind("drugbank", DRUGBANK)
    g.bind("unichem", UNICHEM)
    g.bind("rdf", RDF)
    g.bind("rdfs", RDFS)

    for concept in concepts:
        # Create main compound URI using UniChem ID
        compound_uri = UNICHEM[concept.primary_id]

        # Add basic properties
        g.add((compound_uri, RDF.type, AIDPAIS.Compound))
        g.add((compound_uri, RDFS.label, Literal(concept.primary_label)))  # Use primary_label, not primary_id
        g.add((compound_uri, AIDPAIS.conceptType, Literal(concept.concept_type.value)))

        # Add confidence score
        if concept.confidence_score > 0:
            g.add((compound_uri, AIDPAIS.confidenceScore, Literal(concept.confidence_score)))

        # Add categories
        for category in concept.categories:
            g.add((compound_uri, AIDPAIS.hasCategory, Literal(category)))

        # Add cross-references as separate triples
        for identifier in concept.identifiers:
            if identifier.source == KnowledgeSource.CHEMBL:
                chembl_uri = CHEMBL[identifier.identifier]
                g.add((compound_uri, AIDPAIS.hasChEMBLId, chembl_uri))
                g.add((chembl_uri, RDFS.label, Literal(identifier.label or "")))
                g.add((chembl_uri, AIDPAIS.sameAs, compound_uri))

            elif identifier.source == KnowledgeSource.PUBCHEM:
                pubchem_uri = PUBCHEM[identifier.identifier]
                g.add((compound_uri, AIDPAIS.hasPubChemId, pubchem_uri))
                g.add((pubchem_uri, RDFS.label, Literal(identifier.label or "")))
                g.add((pubchem_uri, AIDPAIS.sameAs, compound_uri))

            elif identifier.source == KnowledgeSource.DRUGBANK:
                drugbank_uri = DRUGBANK[identifier.identifier]
                g.add((compound_uri, AIDPAIS.hasDrugBankId, drugbank_uri))
                g.add((drugbank_uri, RDFS.label, Literal(identifier.label or "")))
                g.add((drugbank_uri, AIDPAIS.sameAs, compound_uri))

            # Add generic cross-reference
            g.add((compound_uri, AIDPAIS.hasIdentifier,
                  Literal(f"{identifier.source.value}:{identifier.identifier}")))

        # Add synonyms
        for synonym in concept.synonyms:
            g.add((compound_uri, AIDPAIS.hasSynonym, Literal(synonym)))

        # Add definitions
        for definition in concept.definitions:
            g.add((compound_uri, AIDPAIS.hasDefinition, Literal(definition)))

    return g

def save_compound_rdf(concepts: List[UnifiedConcept], output_path: str, format: str = "turtle"):
    """
    Convert concepts to RDF and save to file.

    Args:
        concepts: List of UnifiedConcept objects
        output_path: Path to save the RDF file
        format: RDF serialization format ('turtle', 'xml', 'json-ld', etc.)
    """
    g = create_rdf_graph_for_compounds(concepts)

    # Map format names
    format_mapping = {
        'turtle': 'ttl',
        'ttl': 'ttl',
        'xml': 'xml',
        'rdf': 'xml',
        'json-ld': 'json-ld',
        'nt': 'nt'
    }

    output_format = format_mapping.get(format.lower(), 'ttl')

    # Ensure output directory exists
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    # Serialize and save
    g.serialize(output_path, format=output_format)
    print(f"Saved {len(concepts)} compounds to {output_path} in {format} format")
    print(f"Graph contains {len(g)} triples")

def merge_with_existing_graph(new_graph: Graph, existing_graph_path: str, output_path: str):
    """
    Merge new compound data with existing knowledge graph.

    Args:
        new_graph: RDF graph with new compound data
        existing_graph_path: Path to existing KG file
        output_path: Path to save merged graph
    """
    # Load existing graph
    existing_g = Graph()
    existing_g.parse(existing_graph_path, format='turtle')

    print(f"Existing graph: {len(existing_g)} triples")

    # Merge graphs
    existing_g += new_graph

    print(f"Merged graph: {len(existing_g)} triples")

    # Save merged graph
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    existing_g.serialize(output_path, format='turtle')

    print(f"Merged knowledge graph saved to {output_path}")

print("RDF integration utilities loaded successfully!")

RDF integration utilities loaded successfully!


In [32]:
# Example: Convert UniChem results to RDF
print("=== Converting UniChem Results to RDF ===")

# Get some compound data (using results from earlier searches)
aspirin_concepts = await adapter.search_concepts("CHEMBL25", limit=3)
caffeine_concepts = await adapter.search_concepts("CHEMBL3", limit=3)

all_compounds = aspirin_concepts + caffeine_concepts
print(f"Found {len(all_compounds)} compound concepts")

# Convert to RDF
compound_graph = create_rdf_graph_for_compounds(all_compounds)

# Display some RDF triples
print(f"\nGenerated RDF graph with {len(compound_graph)} triples")
print("\nSample triples:")
for i, (s, p, o) in enumerate(compound_graph):
    if i >= 10:  # Show first 10 triples
        break
    print(f"  {s} {p} {o}")

# Save to file
output_file = "./unichem_compounds.ttl"
save_compound_rdf(all_compounds, output_file)

print(f"\nRDF file saved as: {output_file}")

=== Converting UniChem Results to RDF ===


INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.unichem_adapter:UniChem search for 'CHEMBL25' returned 1 concepts
INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.unichem_adapter:UniChem search for 'CHEMBL3' returned 1 concepts
INFO:aid_pais_knowledgegraph.knowledge_lookup.adapters.unichem_adapter:UniChem search for 'CHEMBL3' returned 1 concepts


Found 2 compound concepts

Generated RDF graph with 162 triples

Sample triples:
  https://www.ebi.ac.uk/unichem/compoundsources/310474 http://www.aid-pais-kg.org/hasIdentifier unichem:210363
  https://www.ebi.ac.uk/unichem/compoundsources/310474 http://www.aid-pais-kg.org/hasCategory hmdb
  https://www.ebi.ac.uk/unichem/compoundsources/161671 http://www.aid-pais-kg.org/hasIdentifier unichem:ZINC000000000053
  https://www.ebi.ac.uk/unichem/compoundsources/161671 http://www.aid-pais-kg.org/hasIdentifier unichem:AIN
  https://www.ebi.ac.uk/unichem/compoundsources/161671 http://www.aid-pais-kg.org/hasCategory dailymed
  https://www.ebi.ac.uk/unichem/compoundsources/161671 http://www.aid-pais-kg.org/hasCategory gtopdb
  https://www.ebi.ac.uk/unichem/compoundsources/161671 http://www.aid-pais-kg.org/hasCategory pubchem_tpharma
  https://www.ebi.ac.uk/unichem/compoundsources/161671 http://www.aid-pais-kg.org/hasIdentifier unichem:LYSINE ACETYLSALICYLATE
  https://www.ebi.ac.uk/unichem/compou

In [33]:
# Example: Merge with Existing Knowledge Graph
print("=== Merging with Existing Knowledge Graph ===")

# Path to your existing symptom knowledge graph
existing_kg_path = "../../converted_formats/SymptomGraph_v1.ttl"
merged_output_path = "./merged_knowledge_graph.ttl"

try:
    # Merge compound data with existing KG
    merge_with_existing_graph(compound_graph, existing_kg_path, merged_output_path)
    print("Successfully merged compound data with existing knowledge graph!")

    # Verify the merge by loading and checking
    merged_g = Graph()
    merged_g.parse(merged_output_path, format='turtle')

    # Count compounds in merged graph
    compound_count = 0
    for s, p, o in merged_g.triples((None, RDF.type, AIDPAIS.Compound)):
        compound_count += 1

    print(f"Merged graph now contains {compound_count} compounds")
    print(f"Total triples in merged graph: {len(merged_g)}")

except FileNotFoundError:
    print(f"Existing KG file not found at {existing_kg_path}")
    print("You can merge manually using the merge_ttls.py script:")
    print(f"python ../../scripts/merge_ttls.py -o {merged_output_path} {existing_kg_path} {output_file}")
except Exception as e:
    print(f"Error during merge: {e}")
    print("You can merge manually using the rdflib_manager.py script:")
    print(f"python ../../scripts/database_connectors/rdflib_manager.py --merge-graphs {existing_kg_path} {output_file} --output {merged_output_path}")

=== Merging with Existing Knowledge Graph ===
Existing graph: 1607 triples
Merged graph: 1769 triples
Merged knowledge graph saved to ./merged_knowledge_graph.ttl
Successfully merged compound data with existing knowledge graph!
Merged graph now contains 2 compounds
Total triples in merged graph: 1769
Merged graph now contains 2 compounds
Total triples in merged graph: 1769


In [34]:
# Example: Query the Knowledge Graph
print("=== Querying the Knowledge Graph ===")

def query_compound_cross_references(graph: Graph, compound_id: str):
    """
    Query for compound cross-references in the knowledge graph by UCI ID.
    """
    query = f"""
    PREFIX aidpais: <http://www.aid-pais-kg.org/>
    PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
    PREFIX unichem: <https://www.ebi.ac.uk/unichem/compoundsources/>

    SELECT ?compound ?label ?identifier
    WHERE {{
        ?compound rdfs:label ?label .
        FILTER(?label = "UCI_{compound_id}")
        ?compound aidpais:hasIdentifier ?identifier .
    }}
    """

    results = graph.query(query)
    return results

# Query for aspirin in the merged graph (using UCI ID)
if 'merged_g' in locals():
    aspirin_results = query_compound_cross_references(merged_g, "161671")  # UCI ID for aspirin

    print("Aspirin cross-references in knowledge graph:")
    for row in aspirin_results:
        print(f"Compound: {row.compound}")
        print(f"Label: {row.label}")
        print(f"Identifier: {row.identifier}")
        print("---")
else:
    print("Merged graph not available - load it first:")
    print("merged_g = Graph()")
    print('merged_g.parse("./merged_knowledge_graph.ttl", format="turtle")')

=== Querying the Knowledge Graph ===
Aspirin cross-references in knowledge graph:
Compound: https://www.ebi.ac.uk/unichem/compoundsources/161671
Label: UCI_161671
Identifier: unichem:11126-35-5
---
Compound: https://www.ebi.ac.uk/unichem/compoundsources/161671
Label: UCI_161671
Identifier: unichem:15195166
---
Compound: https://www.ebi.ac.uk/unichem/compoundsources/161671
Label: UCI_161671
Identifier: unichem:15365
---
Compound: https://www.ebi.ac.uk/unichem/compoundsources/161671
Label: UCI_161671
Identifier: unichem:159662
---
Compound: https://www.ebi.ac.uk/unichem/compoundsources/161671
Label: UCI_161671
Identifier: unichem:161671
---
Compound: https://www.ebi.ac.uk/unichem/compoundsources/161671
Label: UCI_161671
Identifier: unichem:20038075
---
Compound: https://www.ebi.ac.uk/unichem/compoundsources/161671
Label: UCI_161671
Identifier: unichem:22360
---
Compound: https://www.ebi.ac.uk/unichem/compoundsources/161671
Label: UCI_161671
Identifier: unichem:2244
---
Compound: https://